# Analisis de los Factores que Influyeron en la Supervivencia de los Pasajeros del Titanic mediante Machine Learning

Este cuaderno sigue la metodologia CRISP-ML para analizar la supervivencia de los pasajeros del Titanic. El modelo principal es una Regresion Logistica para predecir `Survived`; adicionalmente se incluye una Regresion Lineal para predecir `Fare` con fines comparativos y academicos.

## 1. Comprension del problema

**Pregunta de investigacion:** ¿Cuales fueron los factores que mas influyeron en la probabilidad de supervivencia de los pasajeros del Titanic?

**Variable objetivo:** `Survived`, donde 0 significa que el pasajero no sobrevivio y 1 significa que sobrevivio.

**Hipotesis:** variables como sexo, edad, clase, tarifa y tamano de familia influyeron significativamente en la supervivencia.

In [ ]:
from pathlib import Path
import sys

if Path.cwd().name == 'notebooks':
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.titanic_pipeline import FEATURES, load_data, prepare_features, train_and_evaluate

sns.set_theme(style='whitegrid')

## 2. Comprension de los datos

In [ ]:
df = load_data(PROJECT_ROOT / 'data' / 'titanic.csv')
df.head()

In [ ]:
df.info()
df.describe(include='all')

In [ ]:
df.isna().sum().sort_values(ascending=False)

## 3. Exploracion de datos

Se revisa la relacion entre supervivencia, sexo, clase, edad y tarifa.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.countplot(data=df, x='Survived', ax=axes[0, 0])
sns.barplot(data=df, x='Sex', y='Survived', ax=axes[0, 1])
sns.barplot(data=df, x='Pclass', y='Survived', ax=axes[1, 0])
sns.histplot(data=df, x='Age', hue='Survived', kde=True, ax=axes[1, 1])
axes[0, 0].set_title('Distribucion de supervivencia')
axes[0, 1].set_title('Supervivencia por sexo')
axes[1, 0].set_title('Supervivencia por clase')
axes[1, 1].set_title('Edad segun supervivencia')
plt.tight_layout()

## 4. Preparacion e ingenieria de caracteristicas

Se crean `FamilySize = SibSp + Parch + 1` e `IsAlone`, que identifica si el pasajero viajaba solo.

In [ ]:
prepared = prepare_features(df)
prepared[FEATURES + ['Survived']].head()

## 5. Construccion de modelos

Se entrenan dos modelos:

- Regresion Logistica: clasifica si un pasajero sobrevivio.
- Regresion Lineal: predice la tarifa `Fare` para comparar una tarea de regresion con una de clasificacion.

In [ ]:
results = train_and_evaluate(PROJECT_ROOT / 'data' / 'titanic.csv')
results['classification']

In [ ]:
results['regression']

## 6. Interpretacion de resultados

Los coeficientes de la Regresion Logistica permiten identificar que variables aumentan o reducen la probabilidad de supervivencia. Los valores positivos incrementan la probabilidad estimada; los negativos la disminuyen.

In [ ]:
coef = results['coefficients']
coef.head(12)

In [ ]:
plt.figure(figsize=(10, 6))
top_coef = coef.head(10).sort_values('coefficient')
sns.barplot(data=top_coef, x='coefficient', y='feature', palette='viridis')
plt.title('Variables con mayor influencia en la Regresion Logistica')
plt.xlabel('Coeficiente')
plt.ylabel('Variable')
plt.tight_layout()

## 7. Despliegue

Los modelos entrenados se guardan en la carpeta `models/` y las metricas en `reports/`. La aplicacion Streamlit permite ingresar los datos de un pasajero y obtener la probabilidad de supervivencia.

## 8. Conclusiones y recomendaciones

El analisis permite contrastar la hipotesis inicial: el sexo, la clase, la edad, la tarifa y el tamano de familia son variables relevantes para explicar la supervivencia. La Regresion Logistica es adecuada porque la variable objetivo es binaria. La Regresion Lineal se usa como ejercicio comparativo para una variable continua (`Fare`) y no reemplaza al modelo principal de supervivencia.